In [14]:
from transformers import BertTokenizerFast, BertConfig, BertPreTrainedModel, BertModel, get_scheduler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from Levenshtein import ratio
from tqdm.auto import tqdm

class BertJointHead(BertPreTrainedModel):
    def __init__(self, config):
        super().__init__(config)
        self.bert = BertModel(config)
        hidden = config.hidden_size
        self.match_head = nn.Linear(hidden + 1, config.num_labels)
        self.idA_head = nn.Linear(hidden, config.num_labels_entity)
        self.idB_head = nn.Linear(hidden, config.num_labels_entity)
        self.post_init()

    def forward(self, input_ids, attention_mask=None, token_type_ids=None, edit_dist=None):
        out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            return_dict=True
        )
        pooled = out.pooler_output
        ed = edit_dist.unsqueeze(1).to(pooled.dtype)
        feats = torch.cat([pooled, ed], dim=1)
        logits_match = self.match_head(feats)
        logits_idA = self.idA_head(pooled)
        logits_idB = self.idB_head(pooled)
        return logits_match, logits_idA, logits_idB


In [17]:
import pandas as pd
import yaml
import os
import torch

base_dir = os.path.abspath(os.path.join("", '..'))
cfg = yaml.safe_load(open(os.path.join(base_dir, 'configs', 'config.yaml')))
train_csv = os.path.join(base_dir, cfg['data']['input_csv'])
output_dir = cfg['training']['output_dir']
os.makedirs(output_dir, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

df = pd.read_csv(train_csv)
num_entities = df[['clusterA','clusterB']].max().max() + 1

tokenizer = BertTokenizerFast.from_pretrained(cfg['model']['name_or_path'])
bert_conf = BertConfig.from_pretrained(
    cfg['model']['name_or_path'],
    num_labels=int(cfg['model']['num_labels'])
)
bert_conf.num_labels_entity = int(num_entities)
model = BertJointHead(bert_conf).to(device)

In [18]:
sum(p.numel() for p in model.parameters() if p.requires_grad == True)
# 115.52 M params

115515816